In [1]:
from pathlib import Path

PROJECT_ROOT = next(
    p for p in Path.cwd().parents if p.name == "LaboroTomato"
)

DATA_ROOT = PROJECT_ROOT / "dataset" / "cropped"

assert DATA_ROOT.exists(), f"{DATA_ROOT} not found"

In [2]:
import os
import json
import time
import random
from dataclasses import asdict, dataclass
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

In [3]:
# Config
@dataclass
class TrainConfig:
    data_root: str = str(DATA_ROOT)        # contains train/ and test/
    out_dir: str = "../runs_cpu_local/baseline_mobilenetv2"
    model_name: str = "mobilenet_v2"       # "mobilenet_v2" or "efficientnet_v2_s"
    image_size: int = 224
    batch_size: int = 32
    epochs: int = 30
    lr: float = 3e-4
    weight_decay: float = 1e-4
    num_workers: int = 4
    seed: int = 42
    amp: bool = True                        # mixed precision if CUDA
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
# Reproducibility
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Determinism tradeoff: deterministic can be slower
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [5]:
# Model factory
def build_model(model_name: str, num_classes: int) -> nn.Module:
    model_name = model_name.lower()

    if model_name == "mobilenet_v2":
        weights = models.MobileNet_V2_Weights.IMAGENET1K_V1
        model = models.mobilenet_v2(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        return model

    if model_name == "efficientnet_v2_s":
        weights = models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
        model = models.efficientnet_v2_s(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        return model

    raise ValueError(f"Unknown model_name: {model_name}")

In [6]:
# Train / eval loops
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: str, criterion: nn.Module):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)

    avg_loss = total_loss / max(total, 1)
    acc = correct / max(total, 1)
    return avg_loss, acc


def train_one_epoch(model: nn.Module, loader: DataLoader, device: str,
                    criterion: nn.Module, optimizer: torch.optim.Optimizer,
                    scaler: torch.cuda.amp.GradScaler | None):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += x.size(0)

    avg_loss = total_loss / max(total, 1)
    acc = correct / max(total, 1)
    return avg_loss, acc

In [7]:
# Plotting
def save_curves(history_df: pd.DataFrame, out_path: Path):
    fig = plt.figure()
    plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    plt.plot(history_df["epoch"], history_df["train_acc"], label="train_acc")
    plt.plot(history_df["epoch"], history_df["val_acc"], label="val_acc")
    plt.xlabel("epoch")
    plt.legend()
    plt.tight_layout()
    fig.savefig(out_path)
    plt.close(fig)

In [8]:
# Main
def main():
    cfg = TrainConfig()

    seed_everything(cfg.seed)

    # output directory with timestamp to avoid overwriting
    ts = time.strftime("%Y%m%d-%H%M%S")
    run_dir = Path(cfg.out_dir) / ts
    run_dir.mkdir(parents=True, exist_ok=True)

    # Save config (reproducibility)
    with open(run_dir / "config.json", "w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, indent=2)

    data_root = Path(cfg.data_root)
    train_dir = data_root / "train"
    test_dir = data_root / "test"

    # ImageNet normalization
    imagenet_mean = (0.485, 0.456, 0.406)
    imagenet_std = (0.229, 0.224, 0.225)

    # Light augmentation only (as requested)
    train_tfms = transforms.Compose([
        transforms.Resize((cfg.image_size, cfg.image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(
            brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ])

    eval_tfms = transforms.Compose([
        transforms.Resize((cfg.image_size, cfg.image_size)),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ])

    train_ds = datasets.ImageFolder(root=str(train_dir), transform=train_tfms)
    val_ds = datasets.ImageFolder(root=str(test_dir), transform=eval_tfms)

    with open(run_dir / "class_to_idx.json", "w", encoding="utf-8") as f:
        json.dump(train_ds.class_to_idx, f, indent=2)

    num_classes = len(train_ds.classes)

    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True,
        num_workers=cfg.num_workers, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=True
    )

    model = build_model(cfg.model_name, num_classes=num_classes).to(cfg.device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    use_amp = (cfg.amp and cfg.device.startswith("cuda"))
    scaler = torch.cuda.amp.GradScaler() if use_amp else None

    best_val_acc = -1.0
    history = []

    for epoch in range(1, cfg.epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, cfg.device, criterion, optimizer, scaler)
        val_loss, val_acc = evaluate(model, val_loader, cfg.device, criterion)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        })

        # Save "last" every epoch
        torch.save(
            {"model_state": model.state_dict(), "epoch": epoch,
             "val_acc": val_acc, "config": asdict(cfg)},
            run_dir / "model_last.pt"
        )

        # Save "best"
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(
                {"model_state": model.state_dict(), "epoch": epoch,
                 "val_acc": val_acc, "config": asdict(cfg)},
                run_dir / "model_best.pt"
            )

        print(f"Epoch {epoch:03d}/{cfg.epochs} | "
              f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
              f"val loss {val_loss:.4f} acc {val_acc:.4f} | best {best_val_acc:.4f}")

    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / "history.csv", index=False)
    save_curves(history_df, run_dir / "curves.png")

    print(f"\nSaved run artifacts to: {run_dir.resolve()}")
    print(f"Best val acc: {best_val_acc:.4f}")

In [9]:
if __name__ == "__main__":
    main()

Epoch 001/30 | train loss 0.2516 acc 0.8991 | val loss 1.0756 acc 0.8066 | best 0.8066
Epoch 002/30 | train loss 0.1889 acc 0.9226 | val loss 1.0106 acc 0.8030 | best 0.8066
Epoch 003/30 | train loss 0.1735 acc 0.9266 | val loss 1.0963 acc 0.8197 | best 0.8197
Epoch 004/30 | train loss 0.1489 acc 0.9408 | val loss 1.3000 acc 0.8126 | best 0.8197
Epoch 005/30 | train loss 0.1437 acc 0.9432 | val loss 1.2375 acc 0.8080 | best 0.8197
Epoch 006/30 | train loss 0.1285 acc 0.9490 | val loss 1.2777 acc 0.7995 | best 0.8197
Epoch 007/30 | train loss 0.1191 acc 0.9521 | val loss 1.5643 acc 0.8130 | best 0.8197
Epoch 008/30 | train loss 0.1048 acc 0.9582 | val loss 1.2714 acc 0.7999 | best 0.8197
Epoch 009/30 | train loss 0.0950 acc 0.9650 | val loss 1.3592 acc 0.8087 | best 0.8197
Epoch 010/30 | train loss 0.0829 acc 0.9679 | val loss 1.5650 acc 0.8101 | best 0.8197
Epoch 011/30 | train loss 0.0799 acc 0.9690 | val loss 1.2894 acc 0.8038 | best 0.8197
Epoch 012/30 | train loss 0.0794 acc 0.9682

In [6]:
!jupyter nbconvert --to html 02_train_baseline.ipynb


[NbConvertApp] Converting notebook 02_train_baseline.ipynb to html
[NbConvertApp] Writing 322664 bytes to 02_train_baseline.html
